# Carga de datos para las encuestas de uso de tiempo 
## Años: 2003, 2011 y 2024

### Librerias

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

### Rutas
* Directorio ficheros de entrada
  *   ficheros de datos
  *   ficheros de variables
* Directorio para los datasets limpios

In [3]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "Notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / "CleanDataSets").exists():
    candidates = [parent for parent in PROJECT_DIR.parents if (parent / "CleanDataSets").exists()]
    if not candidates:
        raise FileNotFoundError("No se ha podido localizar la carpeta raiz V2.0")
    PROJECT_DIR = candidates[0]
RAW_DIR = PROJECT_DIR / "RawDataSets"
OUTPUT_CSV = PROJECT_DIR / "CleanDataSets"

RAW_FILES = {
    'eut2003':{
        "CH": RAW_DIR / "eut2003/datos ch.txt",
        "MH": RAW_DIR / "eut2003/datos mh.txt",
        "CI": RAW_DIR / "eut2003/datos ci.txt",
        "DI": RAW_DIR / "eut2003/datos di.txt",
    },
    'eut2011':{
        "I": RAW_DIR / "eut2011/eut2011i.data.csv",
        "D": RAW_DIR / "eut2011/eut2011d.data.csv",
    },
    'eut2024':{
        "I": RAW_DIR / "eut2024/eut2024i.data.csv",
        "DA1": RAW_DIR / "eut2024/eut2024da1.data.csv",
        "DA2": RAW_DIR / "eut2024/eut2024da2.data.csv",
    }
}

VAR_FILES = {
    'eut2003':{},
    'eut2011':{
        "I": RAW_DIR / "eut2011/eut2011i.var.csv",
        "D": RAW_DIR / "eut2011/eut2011d.var.csv",
    },
    'eut2024':{
        "I": RAW_DIR / "eut2024/eut2024i.var.csv",
        "DA1": RAW_DIR / "eut2024/eut2024da1.var.csv",
        "DA2": RAW_DIR / "eut2024/eut2024da2.var.csv",
    }

}

### Constantes de traducción entre diferentes años y/o códigos.

In [4]:

CATALUNYA_CODE = "09"

ENCUESTA_YEAR = {
    'eut2003': 2003,
    'eut2011': 2011,
    'eut2024': 2024
} 

SEX_MAP = {
    "1": "Hombre",
    "6": "Mujer",
}

HEALTH_MAP = {
    "1": "Muy bueno",
    "2": "Bueno",
    "3": "Aceptable",
    "4": "Malo",
    "5": "Muy malo",
}

STUDY_MAP_2003_2011 = {
    "01": "Primaria o inferior",        # "No sabe leer/escribir"
    "02": "Primaria o inferior",        # "Sabe leer, menos 5 años de escuela",
    "03": "Primaria o inferior",        # "Sin completar EGB/ESO",
    "04": "Sec. primera etapa",         # "Bachiller elemental/ESO",
    "05": "Sec. segunda etapa",         # "Bachiller superior/BUP/COU",
    "06": "Sec. segunda etapa",         # "FP grado medio",
    "07": "Educación superior",         # "FP grado superior",
    "08": "Educación superior",         # "Diplomatura/Ing. Técnica",
    "09": "Educación superior",         # "Licenciatura/Arquitectura/Ing.",
    "10": "Educación superior",         # "Doctorado"
}

STUDY_MAP_2024 = {
    "1": "Primaria o inferior",
    "2": "Sec. primera etapa",
    "3": "Sec. segunda etapa",
    "4": "Educación superior",
}

STUDY_MAP = {
    'eut2003' : STUDY_MAP_2003_2011,
    'eut2011' : STUDY_MAP_2003_2011,
    'eut2024' : STUDY_MAP_2024
}

LABOR_STATUS_MAP = {
    "eut2003": {
        "01": "Ocupado",
        "02": "Ocupado",
        "03": "Ocupado",
        "04": "Ocupado",
        "05": "Desocupado",
        "06": "Otra inactividad",  # Estudiante o en formación
        "07": "Jubilado o prejubilado",
        "08": "Otra inactividad",  # Incapacidad
        "09": "Otra inactividad",  # Otras pensiones
        "10": "Otra inactividad",  # Tareas del hogar
        "11": "Otra inactividad",  # Voluntariado
        "12": "Otra inactividad",
    },
    "eut2011": {
        "1": "Ocupado",
        "2": "Desocupado",
        "3": "Otra inactividad",  # Estudiante
        "4": "Jubilado o prejubilado",
        "5": "Otra inactividad",  # Incapacidad
        "6": "Otra inactividad",  # Viudedad u orfandad
        "7": "Otra inactividad",  # Voluntariado
        "8": "Otra inactividad",  # Tareas del hogar
        "9": "Otra inactividad",
    },
    "eut2024": {
        "1": "Ocupado",
        "2": "Desocupado",
        "3": "Jubilado o prejubilado",
        "4": "Otra inactividad",
    },
}

STRATUM_MAP_2003 = {
    "0": "Gran ciudad / Área densa",        # Barcelona
    "1": "Gran ciudad / Área densa",        # Capitales de provincia
    "2": "Gran ciudad / Área densa",        # Municipios de más de 100000 habitantes
    "3": "Ciudad media / Área semidensa",   # Municipios entre 50.000 y 100.000 habitantes
    "4": "Ciudad media / Área semidensa",   # Municipios entre 20.000 y 50.000 habitantes
    "5": "Municipio pequeño / Área rural",  # Municipios entre 10.000 y 20.000 habitantes
    "6": "Municipio pequeño / Área rural",  # Municipios de menos de 10.000 habitantes
}

STRATUM_MAP_2011 = {
    "1": "Municipio pequeño / Área rural",
    "2": "Ciudad media / Área semidensa",
    "3": "Gran ciudad / Área densa",
}

STRATUM_MAP_2024 = {
    "1": "Gran ciudad / Área densa",
    "2": "Ciudad media / Área semidensa",
    "3": "Municipio pequeño / Área rural",
}

STRATUM_MAP = {
    'eut2003' : STRATUM_MAP_2003,
    'eut2011' : STRATUM_MAP_2011,
    'eut2024' : STRATUM_MAP_2024
}

TRIMESTER_MAP_2003_2024 = {
    "1": "Ene-Mar",
    "2": "Abr-Jun",
    "3": "Jul-Sep",
    "4": "Oct-Dic",
}

TRIMESTER_MAP_2011 = {
    1: "Ene-Mar", 2: "Ene-Mar", 3: "Ene-Mar",
    4: "Abr-Jun", 5: "Abr-Jun", 6: "Abr-Jun",
    7: "Jul-Sep", 8: "Jul-Sep", 9: "Jul-Sep",
    10: "Oct-Dic", 11: "Oct-Dic", 12: "Oct-Dic",
}

TRIMESTER_MAP = {
    'eut2003' : TRIMESTER_MAP_2003_2024,
    'eut2011' : TRIMESTER_MAP_2011,
    'eut2024' : TRIMESTER_MAP_2003_2024
}

DAY_TYPE_MAP = {
    "eut2003": {"1": "Día habitual", "6": "Día inusual"},
    "eut2011": {"1": "Día habitual", "2": "Día inusual"},
    "eut2024": {"1": "Día habitual", "2": "Día inusual"},
}

DAY_MAP_2003 = {
    "1": "Lunes-Jueves",
    "2": "Lunes-Jueves",
    "3": "Lunes-Jueves",
    "4": "Lunes-Jueves",
    "5": "Viernes-Domingo",
    "6": "Viernes-Domingo",
    "7": "Viernes-Domingo",
}

DAY_MAP_2011 = {
    0: "Lunes-Jueves",
    1: "Lunes-Jueves",
    2: "Lunes-Jueves",
    3: "Lunes-Jueves",
    4: "Viernes-Domingo",
    5: "Viernes-Domingo",
    6: "Viernes-Domingo",
}

DAY_MAP_2024 = {
    "1": "Lunes-Jueves",
    "2": "Viernes-Domingo",
}

DAY_MAP = {
    'eut2003': DAY_MAP_2003,
    'eut2011': DAY_MAP_2011,
    'eut2024': DAY_MAP_2024
}

AGE_GROUP_MAP = {
    "1": "10-24 años",
    "2": "25-44 años",
    "3": "45-64 años",
    "4": "65 o más",
}

OUTPUT_COLUMNS = [
    "encuesta_year", "hogar_uid", "persona_uid", "id_hogar", "n_pers", 
    "tramo_edad", "sexo", "nivel_estudios","situacion_laboral", "estrato", "estado_salud", 
    "trimestre", "dia_semana", "tipo_dia", "internet_medicion", "factor_elevacion"
]



### Definición de las actividades, códigos y correspondencias

In [5]:
MAIN_ACTIVITY_BY_FIRST_DIGIT = {
    "0": "min_cuidados_personales",
    "1": "min_trabajo_remunerado",
    "2": "min_estudios",
    "3": "min_hogar_familia",
    "4": "min_voluntario_reuniones",
    "5": "min_vida_social_diversion",
    "6": "min_deportes_airelibre",
    "7": "min_aficiones_informatica",
    "8": "min_medios_comunicacion",
    "9": "min_trayectos_noespec",
}

SUBACTIVITY_BY_FIRST_TWO_DIGITS = {
    # Cuidados personales
    "01": "min_cp_dormir",
    "02": "min_cp_comer_beber",
    "03": "min_cp_higiene_salud",

    # Trabajo
    "11": "min_tr_actividad_laboral",
    "12": "min_tr_actividades_relacionadas",
    "13": "min_tr_actividades_relacionadas",

    # Estudios
    "21": "min_est_clases_formacion",
    "22": "min_est_estudio_tiempo_libre",

    # Hogar y familia
    "31": "min_hf_actividades_culinarias",
    "32": "min_hf_mantenimiento_hogar",
    "33": "min_hf_ropa",
    "34": "min_hf_jardineria_animales",
    "35": "min_hf_construccion_reparaciones",
    "36": "min_hf_compras_servicios",
    "37": "min_hf_gestiones_hogar",
    "38": "min_hf_cuidado_menores",
    "39": "min_hf_cuidado_adultos",

    # Voluntariado y participación
    "41": "min_vr_voluntariado_organizado",
    "42": "min_vr_ayuda_otros_hogares",
    "43": "min_vr_participacion_religion",

    # Vida social
    "51": "min_vs_vida_social",
    "52": "min_vs_diversion_cultura",
    "53": "min_vs_ocio_pasivo",

    # Deporte
    "61": "min_da_ejercicio_fisico",
    "62": "min_da_ejercicio_productivo",
    "63": "min_da_actividades_relacionadas",

    # Aficiones
    "71": "min_ai_arte_aficiones",
    "72": "min_ai_informatica",
    "73": "min_ai_juegos",

    # Medios
    "81": "min_mc_lectura",
    "82": "min_mc_television_video",
    "83": "min_mc_radio_grabaciones",

    # Trayectos
    "91": "min_tray_trabajo",
    "92": "min_tray_estudios",
    "93": "min_tray_hogar_familia",
    "94": "min_tray_voluntariado",
    "95": "min_tray_vida_social",
    "96": "min_tray_otro_ocio",
    "98": "min_tray_cambio_municipio",
    "90": "min_tray_otros_no_especificado",
}

SECONDS_TO_MINUTES = {
    # Categorías principales actuales
    "curaper": "min_cuidados_personales",
    "treball": "min_trabajo_remunerado",
    "estudi": "min_estudios",
    "llarfam": "min_hogar_familia",
    "volreun": "min_voluntario_reuniones",
    "vidsoc": "min_vida_social_diversion",
    "esport": "min_deportes_airelibre",
    "aficc": "min_aficiones_informatica",
    "mitjcom": "min_medios_comunicacion",
    "trajecte": "min_trayectos_noespec",

    # Cuidados personales
    "dormir": "min_cp_dormir",
    "menjar": "min_cp_comer_beber",
    "altrecur": "min_cp_higiene_salud",

    # Trabajo
    "trebprin": "min_tr_actividad_laboral",
    "altretreb": "min_tr_actividades_relacionadas",

    # Estudios
    "estudis": "min_est_clases_formacion",
    "estudis_tll": "min_est_estudio_tiempo_libre",

    # Hogar y familia
    "culin": "min_hf_actividades_culinarias",
    "mantllar": "min_hf_mantenimiento_hogar",
    "confecc": "min_hf_ropa",
    "jardi": "min_hf_jardineria_animales",
    "repar": "min_hf_construccion_reparaciones",
    "compra": "min_hf_compras_servicios",
    "gestio": "min_hf_gestiones_hogar",
    "curanen": "min_hf_cuidado_menores",
    "curadul": "min_hf_cuidado_adultos",

    # Voluntariado
    "organit": "min_vr_voluntariado_organizado",
    "altllar": "min_vr_ayuda_otros_hogares",
    "particip": "min_vr_participacion_religion",

    # Vida social
    "sociabil": "min_vs_vida_social",
    "cultura": "min_vs_diversion_cultura",
    "lleurepa": "min_vs_ocio_pasivo",

    # Deporte
    "exfis": "min_da_ejercicio_fisico",
    "exprod": "min_da_ejercicio_productivo",
    "altresp": "min_da_actividades_relacionadas",

    # Aficiones y medios
    "art": "min_ai_arte_aficiones",
    "informat": "min_ai_informatica",
    "jocs": "min_ai_juegos",
    "lectura": "min_mc_lectura",
    "Tele": "min_mc_television_video",
    "radio": "min_mc_radio_grabaciones",

    # Trayectos
    "desplaç_treb": "min_tray_trabajo",
    "desplaç_est": "min_tray_estudios",
    "desplaç_llarfam": "min_tray_hogar_familia",
    "desplaç_vol": "min_tray_voluntariado",
    "desplaç_vidsoc": "min_tray_vida_social",
    "desplaç_tll": "min_tray_otro_ocio",
    "desplaç_muni": "min_tray_cambio_municipio",
    "desplaç_fi": "min_tray_otros_no_especificado",
}

INTERNET_ACTIVITY_BY_DIGIT = {
    digit: f"min_internet_{col.removeprefix('min_')}"
    for digit, col in MAIN_ACTIVITY_BY_FIRST_DIGIT.items()
}

INTERNET_COLS = [
    "min_internet_total",
    *INTERNET_ACTIVITY_BY_DIGIT.values(),
]

MAIN_MINUTE_COLS_2003_2024  = list(MAIN_ACTIVITY_BY_FIRST_DIGIT.values())
SUB_MINUTE_COLS_2003_2024   = list(dict.fromkeys(SUBACTIVITY_BY_FIRST_TWO_DIGITS.values()))
MINUTE_COLS_2003_2024       = MAIN_MINUTE_COLS_2003_2024 + SUB_MINUTE_COLS_2003_2024 + INTERNET_COLS



TIME_COLS = {
    'eut2003':{
        'main_minute_cols'  : MAIN_MINUTE_COLS_2003_2024,
        'sub_minute_cols'   : SUB_MINUTE_COLS_2003_2024,
        'minute_cols'       : MINUTE_COLS_2003_2024
    },
    'eut2011':{
        'main_minute_cols'  : list(SECONDS_TO_MINUTES.values())[:10],
        'minute_cols'       : list(dict.fromkeys(SECONDS_TO_MINUTES.values())) + INTERNET_COLS,
    },
    'eut2024':{
        'main_minute_cols'  : MAIN_MINUTE_COLS_2003_2024,
        'sub_minute_cols'   : SUB_MINUTE_COLS_2003_2024,
        'minute_cols'       : MINUTE_COLS_2003_2024
    }
}


### Métodos auxiliares para diversos años

In [6]:
def to_colspecs(specs_1based):
    # Resta uno a la posición inicial, los incidices empiezan en 0 en python.
    return [(start - 1, end) for start, end in specs_1based]

def read_fixed_width(path, specs_1based, names):
    return pd.read_fwf(
        path,
        colspecs=to_colspecs(specs_1based),
        names=names,
        dtype=str,
        encoding="latin1",
    )

def as_number(series):
    return pd.to_numeric(series, errors="coerce")

def empty_minutes_record():
    return {col: 0 for col in TIME_COLS['eut2003']['minute_cols']}

def normalize_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lstrip("0")
        .replace({"": np.nan, "nan": np.nan})
    )

def load_tsv(path):
    
    df = pd.read_csv(path, sep="\t", dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan})
    
    return df

def read_var_file(path):
    df = pd.read_csv(path, dtype=str, engine="python", on_bad_lines="skip")
    df.columns = [c.strip() for c in df.columns]
   
    return df


### Métodos propios para la encuesta del 2003

In [7]:
def load_home_2003_data():
    ch = read_fixed_width(
        RAW_FILES['eut2003']["CH"],
        specs_1based=[(1, 5), (6, 7), (8, 8), (11,11)],
        names=["id_hogar", "CAUT", "estrato_code", "trimestre"],
    )
    ch["trimestre"] = ch["trimestre"].map(TRIMESTER_MAP['eut2003'])
    ch = ch[ch["CAUT"].eq(CATALUNYA_CODE)].copy()
    ch["estrato"] = ch["estrato_code"].map(STRATUM_MAP['eut2003'])

    return ch

def load_members_home_2003_data():
    mh = read_fixed_width(
        RAW_FILES['eut2003']["MH"],
        specs_1based=[(1, 5), (6, 7), (9, 9), (30, 32), (33, 34)],
        names=["id_hogar", "n_pers", "sexo_code", "edad", "situacion_laboral_code"],
    )
    mh["sexo"] = mh["sexo_code"].map(SEX_MAP)
    mh["edad"] = as_number(mh["edad"])
    mh["tramo_edad"] = pd.cut( mh["edad"], bins=[9, 24, 44, 64, 200], labels=list(AGE_GROUP_MAP.values()) )
    mh["situacion_laboral"] = mh["situacion_laboral_code"].map(LABOR_STATUS_MAP["eut2003"])

    return mh

def load_individual_2003_data():
    ci = read_fixed_width(
        RAW_FILES['eut2003']["CI"],
        specs_1based=[(1, 5), (6, 7), (449, 450), (451, 451)],
        names=["id_hogar", "n_pers", "estudios_code", "salud_code"],
    )
    ci["nivel_estudios"] = ci["estudios_code"].map( STUDY_MAP['eut2003'] )
    ci["estado_salud"] = ci["salud_code"].map(HEALTH_MAP)

    return ci

def parse_diary_file_2003( cat_houselist ):
    rows = []
    with RAW_FILES['eut2003']["DI"].open(encoding="latin1") as fh:
        for line in fh:
            id_hogar = line[0:5]
            if id_hogar not in cat_houselist:
                continue

            factor_hogar = pd.to_numeric(line[1896:1907], errors="coerce") / 1_000_000
            factor_persona = pd.to_numeric(line[1907:1913], errors="coerce") / 100_000        

            rec = {
                "id_hogar": id_hogar,
                "n_pers": line[5:7],
                "dia_semana": DAY_MAP["eut2003"].get(line[9:10]),
                "tipo_dia": DAY_TYPE_MAP["eut2003"].get(line[1886:1887]),
                "factor_elevacion": (factor_hogar * factor_persona) / 2,
            }
            rec.update(empty_minutes_record())
            rec["internet_medicion"] = "Internet"

            for i in range( 144 ):
                start = 10 + i * 13
                aprin = line[start:start + 4]
                internet_code = line[start + 10:start + 11]
                if len( aprin ) < 4 or not aprin.strip():
                    continue
                
                if internet_code == "1":
                    rec["min_internet_total"] += 10

                    internet_col = INTERNET_ACTIVITY_BY_DIGIT.get(aprin[:1])
                    if internet_col:
                        rec[internet_col] += 10

                main_col = MAIN_ACTIVITY_BY_FIRST_DIGIT.get(aprin[:1])
                if main_col:
                    rec[main_col] += 10

                sub_col = SUBACTIVITY_BY_FIRST_TWO_DIGITS.get(aprin[:2])
                if sub_col:
                    rec[sub_col] += 10

            rows.append(rec)

    return pd.DataFrame(rows)


### Métodos propios para la encuesta del 2011

In [8]:
def build_individual_2011( individual_df ):

    base = individual_df.rename(columns={"CODI_LLAR": "id_hogar", "CODI_PERSONA": "n_pers"}).copy()

    base["id_hogar"] = normalize_id( base["id_hogar"] )
    base["n_pers"] = normalize_id( base["n_pers"] )
    base["edad"] = pd.to_numeric( base["Edat"], errors="coerce" )
    base["sexo"] = base["Sexe"].map( SEX_MAP )
    base["nivel_estudios"] = base["T6_ESTUDI_ALT"].astype(str).str.zfill(2).map( STUDY_MAP['eut2011'] )
    base["estado_salud"] = base["T6_SALUT"].map( HEALTH_MAP )
    base["tramo_edad"] = pd.cut( base["edad"], bins=[0, 24, 44, 64, 200], labels=list(AGE_GROUP_MAP.values()) ).astype("object")
    base["factor_elevacion"] = pd.to_numeric(base["PES_INDIVIDUS_CALMAR"], errors="coerce") / 1_000
    base["situacion_laboral"] = base["T3_SITUACIO_ACTIV"].map(LABOR_STATUS_MAP["eut2011"])

    return base

def build_diary_2011( diari_df ):

    diary = diari_df.rename(columns={"CODI_LLAR": "id_hogar", "CODI_PERSONA": "n_pers"}).copy()
    diary['id_hogar'] = normalize_id( diary["id_hogar"] )
    diary['n_pers'] = normalize_id( diary["n_pers"] )

    for raw_col, out_col in SECONDS_TO_MINUTES.items():
        diary[out_col] = pd.to_numeric(diary[raw_col], errors="coerce") / 60

    year_survey = pd.to_numeric(diary["T1_DATA_ANY"], errors="coerce") + 2000
    month_survey = pd.to_numeric(diary["T1_DATA_MES"], errors="coerce")
    day_survey = pd.to_numeric(diary["T1_DATA_DIA"], errors="coerce")
    date = pd.to_datetime(
        {"year": year_survey, "month": month_survey, "day": day_survey},
        errors="coerce",
    )
    diary["trimestre"] = month_survey.map(TRIMESTER_MAP['eut2011'])
    diary["dia_semana"] = date.dt.weekday.map(DAY_MAP['eut2011'])
    diary["tipo_dia"] = diary["T7_TIPUS_DIA"].map(DAY_TYPE_MAP["eut2011"])
    diary["estrato"] = diary["DIM_MUN"].map(STRATUM_MAP['eut2011'])

    for col in INTERNET_COLS:
        diary[col] = np.nan

    diary["internet_medicion"] = "No disponible"

    return diary

### Métodos propios para la encuesta del 2024

In [9]:
def build_individual_2024(individual_df):

    base = individual_df.rename(columns={"IDLLAR": "id_hogar", "IDPERSONA": "n_pers"}).copy()
    base["n_pers"] = normalize_id(base["n_pers"])
    base["sexo"] = base["SEXE"].map(SEX_MAP)
    base["tramo_edad"] = base["TRAM_EDAT"].map(AGE_GROUP_MAP)
    base["nivel_estudios"] = base["ESTUDI_ALT"].map(STUDY_MAP['eut2024'])
    base["estado_salud"] = base["SALUT"].map(HEALTH_MAP)
    base["situacion_laboral"] = (
        base["REL_ACT"].fillna("4") # Otra inactividad
        .map(LABOR_STATUS_MAP["eut2024"])
    )

    return base

def build_diary_2024(meta_df):
    
    meta = meta_df.rename(columns={"IDLLAR": "id_hogar", "IDPERSONA": "n_pers"}).copy()
    meta["n_pers"] = normalize_id(meta["n_pers"])
    meta["estrato"] = meta["DEGURBA"].map(STRATUM_MAP['eut2024'])
    meta["trimestre"] = meta["TRIM"].map(TRIMESTER_MAP['eut2024'])
    meta["dia_semana"] = meta["DIA_SETMANA"].map(DAY_MAP['eut2024'])
    meta["tipo_dia"] = meta["TIPUS_DIA"].map(DAY_TYPE_MAP["eut2024"])
    meta["factor_elevacion"] = ( pd.to_numeric(meta["FE_DIARIS"], errors="coerce") / 100_000 )

    return meta

def build_activity_2024(slots_df):

    slots = slots_df.rename(columns={"IDLLAR": "id_hogar", "IDPERSONA": "n_pers"}).copy()
    slots["n_pers"] = normalize_id(slots["n_pers"])

    slots["act_code"] = slots["ACT_PPAL"].astype(str).str.zfill(3)
    slots["main_col"] = slots["act_code"].str[:1].map(MAIN_ACTIVITY_BY_FIRST_DIGIT)
    slots["sub_col"] = slots["act_code"].str[:2].map(SUBACTIVITY_BY_FIRST_TWO_DIGITS)
    slots["internet_flag"] = slots["INTERNET"].eq("1")
    slots["main_digit"] = slots["act_code"].str[:1]

    main_counts = (
        slots.dropna(subset=["main_col"])
        .groupby(["id_hogar", "n_pers", "main_col"])
        .size()
        .mul(10)
        .unstack(fill_value=0)
    )
    sub_counts = (
        slots.dropna(subset=["sub_col"])
        .groupby(["id_hogar", "n_pers", "sub_col"])
        .size()
        .mul(10)
        .unstack(fill_value=0)
    )
    internet_total = (
        slots[slots["internet_flag"]]
        .groupby(["id_hogar", "n_pers"])
        .size()
        .mul(10)
        .rename("min_internet_total")
    )
    internet_activity = (
        slots[slots["internet_flag"]]
        .groupby(["id_hogar", "n_pers", "main_digit"])
        .size()
        .mul(10)
        .unstack(fill_value=0)
        .rename(columns=INTERNET_ACTIVITY_BY_DIGIT)
    )

    activity = main_counts.join(sub_counts, how="outer").fillna(0).reset_index()

    activity = (
        activity
        .set_index(["id_hogar", "n_pers"])
        .join(internet_total, how="left")
        .join(internet_activity, how="left")
        .reset_index()
    )

    activity[INTERNET_COLS] = activity[INTERNET_COLS].fillna(0)
    activity["internet_medicion"] = "Dispositivo electrónico o Internet"

    for col in TIME_COLS['eut2024']['minute_cols']:
        if col not in activity.columns:
            activity[col] = 0
    return activity[[
        "id_hogar", "n_pers", "internet_medicion",
        *TIME_COLS['eut2024']['minute_cols'],
    ]]


### Definición de los métodos de carga para cada año.

In [10]:
def load_2024():

    individual = load_tsv(RAW_FILES['eut2024']["I"])
    diary_meta = load_tsv(RAW_FILES['eut2024']["DA1"])
    diary_slots = load_tsv(RAW_FILES['eut2024']["DA2"])

    person_base = build_individual_2024( individual )
    meta = build_diary_2024( diary_meta )
    activity = build_activity_2024(diary_slots)

    return (
    person_base
    .merge(meta[[
        "id_hogar", "n_pers", "estrato", "trimestre", "dia_semana",
         "tipo_dia", "factor_elevacion",
    ]], on=["id_hogar", "n_pers"], how="inner")
    .merge(activity, on=["id_hogar", "n_pers"], how="inner")
    )


def load_2011():
    
    individual = load_tsv(RAW_FILES['eut2011']["I"])
    diari = load_tsv(RAW_FILES['eut2011']["D"])

    person_base = build_individual_2011( individual )
    diary = build_diary_2011( diari )
                             
    return person_base.merge(
        diary[[
            "id_hogar", "n_pers", "estrato", "dia_semana",
            "tipo_dia", "trimestre", "internet_medicion",
        ] + TIME_COLS['eut2011']['minute_cols']],
        on=["id_hogar", "n_pers"],
        how="inner",
    ).sort_values(["id_hogar", "n_pers"], key=lambda s: pd.to_numeric(s, errors="coerce")).reset_index(drop=True)


def load_2003():

    ch = load_home_2003_data()
    mh = load_members_home_2003_data()
    ci = load_individual_2003_data()

    person_base = (
        mh.merge(ci, on=["id_hogar", "n_pers"], how="inner")
        .merge(ch[["id_hogar", "estrato","trimestre"]], on="id_hogar", how="inner")
    )
    diary = parse_diary_file_2003(set(ch["id_hogar"]))

    return (
        person_base.merge(diary, on=["id_hogar", "n_pers"], how="inner")
        .sort_values(["id_hogar", "n_pers"])
        .reset_index(drop=True)
    )


### Punto de inicio de la carga de datos.

In [11]:
YEAR_LOAD_METHOD = {
    'eut2003': load_2003,
    'eut2011': load_2011,
    'eut2024': load_2024
}

for year, load_method in YEAR_LOAD_METHOD.items():

    data_year = load_method()
    data_year["encuesta_year"] = ENCUESTA_YEAR[year]
    data_year["hogar_uid"]    = data_year["encuesta_year"].astype(str) + "_" + data_year["id_hogar"]
    data_year["persona_uid"]  = data_year["hogar_uid"] + "_" + data_year["n_pers"]
    data_year = data_year[OUTPUT_COLUMNS + TIME_COLS[year]['minute_cols']]

    # Validaciones sobre el identificador de persona y los registros de tiempo.
    assert data_year["persona_uid"].notna().all(), f"{year}: hay identificadores personales vacíos"
    assert not data_year["persona_uid"].duplicated().any(), f"{year}: persona_uid no es único"

    total_minutos = data_year[TIME_COLS[year]["main_minute_cols"]].sum(axis=1)
    assert np.isclose(total_minutos, 1440).all(), f"{year}: algún diario no suma 1.440 minutos"

    OUTPUT_CSV.mkdir(parents=True, exist_ok=True)
    file_name = year + '.csv'
    data_year.to_csv(OUTPUT_CSV / file_name, index=False, encoding="utf-8")
    